# 02 - Fine-tune MiniLM on cleaned full AllNLI with early stopping

Notebook nay chay doc lap tren Kaggle. Model train tren clean train split day du, dung validation de early stopping, roi danh gia test 5k va thoi gian scoring test 5k.

In [1]:
from pathlib import Path

PROJECT_ROOT = Path('/kaggle/working/similarity_search')
GITHUB_REPOSITORY_URL = 'https://github.com/PhDQuang/similarity_search.git'

if not PROJECT_ROOT.exists():
    !git clone {GITHUB_REPOSITORY_URL} {PROJECT_ROOT}

%cd {PROJECT_ROOT}
%pip install -q -r fix/requirements-kaggle.txt
%pip install -q -e fix

Cloning into '/kaggle/working/similarity_search'...
remote: Enumerating objects: 222, done.
remote: Counting objects: 100% (222/222), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 222 (delta 90), reused 185 (delta 53), pack-reused 0 (from 0)
Receiving objects: 100% (222/222), 481.65 KiB | 11.75 MiB/s, done.
Resolving deltas: 100% (90/90), done.
/kaggle/working/similarity_search
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for similarity-search-fix (pyproject.toml) ... done
Note: yo

In [2]:
from pathlib import Path
import shutil

CLEAN_DATA_DIR = Path('fix/data/processed/allnli_70_15_15_clean/pair-class')
KAGGLE_CLEAN_CANDIDATES = [
    Path('/kaggle/input/allnli-70-15-15-clean/pair-class'),
    Path('/kaggle/input/allnli-70-15-15-clean/allnli_70_15_15_clean/pair-class'),
]

def has_clean_data(path: Path) -> bool:
    return all((path / f'{split}.parquet').exists() for split in ('train', 'val', 'test'))

if not has_clean_data(CLEAN_DATA_DIR):
    source = next((path for path in KAGGLE_CLEAN_CANDIDATES if has_clean_data(path)), None)
    if source is not None:
        CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)
        for item in source.iterdir():
            if item.is_file():
                shutil.copy2(item, CLEAN_DATA_DIR / item.name)
    else:
        !python -m similarity_search_fix.data.prepare_allnli_70_15_15_clean --output-dir {CLEAN_DATA_DIR} --seed 42

assert has_clean_data(CLEAN_DATA_DIR), f'Missing clean data: {CLEAN_DATA_DIR}'
print('Using clean data:', CLEAN_DATA_DIR)

README.md: 5.15kB [00:00, 2.27MB/s]
pair-class/train-00000-of-00001.parquet: 100%|█| 69.5M/69.5M [00:03<00:00, 18.1M
pair-class/dev-00000-of-00001.parquet: 100%|█| 1.57M/1.57M [00:00<00:00, 2.56MB/
pair-class/test-00000-of-00001.parquet: 100%|█| 1.61M/1.61M [00:00<00:00, 2.62MB
Generating train split: 100%|█| 942069/942069 [00:00<00:00, 1308140.19 examples/
Generating test split: 100%|██| 19656/19656 [00:00<00:00, 1212882.17 examples/s]
{
  "dataset_name": "sentence-transformers/all-nli",
  "dataset_config": "pair-class",
  "created_at_utc": "2026-07-05T07:23:38.455239+00:00",
  "seed": 42,
  "split_ratios": {
    "train": 0.7,
    "val": 0.15,
    "test": 0.15
  },
  "saved_paths": {
    "train": "fix/data/processed/allnli_70_15_15_clean/pair-class/train.parquet",
    "val": "fix/data/processed/allnli_70_15_15_clean/pair-class/val.parquet",
    "test": "fix/data/processed/allnli_70_15_15_clean/pair-class/test.parquet"
  },
  "row_counts": {
    "train": 686442,
    "val": 147033,
    

In [3]:
from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working/fix_outputs/minilm_clean_full')
MODEL_DIR = Path('/kaggle/working/fix_models/minilm_clean_full')

!python -m similarity_search_fix.models.train_minilm \
  --input-dir {CLEAN_DATA_DIR} \
  --output-dir {OUTPUT_DIR} \
  --model-dir {MODEL_DIR} \
  --num-train-epochs 5 \
  --batch-size 64 \
  --eval-batch-size 128 \
  --learning-rate 5e-6 \
  --eval-steps 1000 \
  --save-steps 1000 \
  --trainer-eval-samples 20000 \
  --early-stopping-patience 2 \
  --early-stopping-threshold 0.0 \
  --metric-for-best-model eval_fixed-allnli-val_spearman_cosine \
  --max-retrieval-queries 1000 \
  --test-sample-size 5000 \
  --seed 42

/kaggle/working/similarity_search/fix/src/similarity_search_fix/models/train_minilm.py:138: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import CosineSimilarityLoss
/kaggle/working/similarity_search/fix/src/similarity_search_fix/models/train_minilm.py:84: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
config_sentence_transformers.json: 100%|████████| 116/116 [00:00<00:00, 823kB/s]
README.md: 10.5kB [00:00, 31.6MB/s]
sentence_bert_config.json: 100%|██████████████| 53.0/53.0 [00:00<00:00, 397kB/s]
config.json: 100%|█████████████████████████████| 612/612 [00:00<00:00, 3.50MB/s]
mode

In [4]:
from pathlib import Path
import json
import shutil

ARTIFACT_DIR = Path('/kaggle/working/artifacts_minilm_clean_full')
if ARTIFACT_DIR.exists():
    shutil.rmtree(ARTIFACT_DIR)
ARTIFACT_DIR.mkdir(parents=True)
shutil.copytree(OUTPUT_DIR, ARTIFACT_DIR / 'outputs')
shutil.copytree(MODEL_DIR, ARTIFACT_DIR / 'model')
zip_path = shutil.make_archive(str(ARTIFACT_DIR), 'zip', ARTIFACT_DIR)
print('Download artifact:', zip_path)

display(json.loads((OUTPUT_DIR / 'metrics.json').read_text()))
display(json.loads((OUTPUT_DIR / 'test5k_performance.json').read_text()))

Download artifact: /kaggle/working/artifacts_minilm_clean_full.zip


{'task': 'entailment-as-semantic-similarity',
 'fixed_dataset': 'AllNLI pair-class full 70/15/15',
 'positive_label': 'entailment',
 'model': {'name': 'Fine-tuned MiniLM',
  'base_model': 'sentence-transformers/all-MiniLM-L6-v2',
  'trained_in_project': True,
  'embedding_dimension': 384,
  'loss': 'CosineSimilarityLoss',
  'score_mapping': {'entailment': 1.0, 'neutral': 0.5, 'contradiction': 0.0}},
 'threshold_selection': {'split': 'val',
  'threshold': 0.6016422510147095,
  'best_f1': 0.709952783631659},
 'pair_classification': {'val': {'threshold': 0.6016422510147095,
   'accuracy': 0.7781518434637122,
   'precision': 0.6286672650824396,
   'recall': 0.8153799019607844,
   'f1': 0.7099527836316589,
   'average_precision': 0.7253473170550009,
   'mean_positive_score': 0.7347835898399353,
   'mean_negative_score': 0.3866136074066162,
   'roc_auc': 0.862371999109107},
  'test': {'threshold': 0.6016422510147095,
   'accuracy': 0.7775252611165924,
   'precision': 0.6281924340295786,
   '

{'sample_rows': 5000,
 'elapsed_seconds': 2.6715839620001134,
 'pairs_per_second': 1871.5488905153818,
 'threshold': 0.6016422510147095,
 'pair_classification': {'threshold': 0.6016422510147095,
  'accuracy': 0.7728,
  'precision': 0.6246537396121884,
  'recall': 0.8072792362768496,
  'f1': 0.7043206663196252,
  'average_precision': 0.73362805445698,
  'mean_positive_score': 0.7313576936721802,
  'mean_negative_score': 0.38909009099006653,
  'roc_auc': 0.8597087357728133}}